In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Mon Mar  3 20:09:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.15              Driver Version: 570.86.15      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 4500 Ada Gene...    Off |   00000000:16:00.0 Off |                  Off |
| 30%   34C    P8              8W /  210W |      18MiB /  24570MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip3 install --upgrade --quiet pip
!pip3 install --upgrade --quiet datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio 
!pip3 install librosa
!pip3 install ipywidgets
!pip3 install deepspeed
!pip3 install mpi4py
!pip3 install soundfile
!pip3 install python-dotenv

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [5]:
from huggingface_hub import login
login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
from datasets import load_dataset, DatasetDict, concatenate_datasets

data = load_dataset("brunopbb/ufcg-labmet-fala-texto-main")

train = data['train']
test = data['test']

In [7]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-medium", language="Portuguese", task="transcribe")


In [8]:
def prepare_dataset(batch):
    # Para carregar e reamostrar dados de áudio de 48kHz para 16kHz
    audio = batch["audio"]

    # Para calcular as características de entrada log-Mel a partir de um array de áudio de entrada
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # Para codificar texto alvo em IDs de rótulos, você precisará de um tokenizador.
    # O tokenizador mapeia cada caractere único no seu texto para um ID numérico.
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    #batch["attention_mask"] = [1] * len(batch["input_features"])
    return batch

In [9]:
data = data.map(prepare_dataset, remove_columns=data.column_names["train"], num_proc=2)

In [10]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
model.generation_config.language = "portuguese"
model.generation_config.task = "transcribe"
model.config.use_cache = False
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

#model.config.dropout = 0.1
#model.config.attention_dropout = 0.1


In [11]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Extrai e preenche as features de entrada.
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(
            input_features, padding="longest", return_tensors="pt"
        )

        #batch["attention_mask"] = batch["input_features"].ne(self.processor.tokenizer.pad_token_id).long()

        # Extrai e preenche os rótulos.
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, padding="longest", return_tensors="pt"
        )

        # Substitui os tokens de preenchimento por -100 para ignorar a perda.
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        # Remove o token de início do decoder, se já adicionado.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch




In [12]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [13]:
import evaluate

metric = evaluate.load("wer")

In [14]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    #Substitua -100 pelo ID do token de preenchimento
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    #Nós não queremos agrupar tokens ao calcular as métricas.
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [17]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="../Fine-Tunning-medium-v1.2 ",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=3e-6,
    max_steps=250,
    gradient_checkpointing = True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=1,
    predict_with_generate=True,
    generation_max_length=50,
    save_steps=50,
    eval_steps=50,
    logging_steps=25 ,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    #push_to_hub = True,
    #deepspeed = "/home/bruno-dev/Documents/projeto-fala-texto/fine-tuning/ds_config.json"
)

In [18]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset = data['train'],
    eval_dataset = data['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/home/fala-texto-server2/anaconda3/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


In [ ]:
model.save_pretrained("/mnt/hd/v1-medium/whisper_finetuned-medium-v1.2")
processor.save_pretrained("/mnt/hd/v1-medium/whisper_finetuned-medium-v1.2")

-- Realizar Avaliação do Treinamento

In [ ]:
results = trainer.evaluate()
print(f"Word Error Rate: {results['eval_wer']}")